# 30c — ALM controls (uniform): regional specificity

PPC silencing (`30`/`31`) moves behaviour; does silencing a *different* frontal region — ALM —
reproduce it, or is the PPC effect regional? Two dedicated ALM control session types, both run
in `uniform`, both compared opto (laser at ALM, `Zapit=True`) vs the interleaved `opto_off`.

- **Unilateral ALM → side bias (§A).** Always the same hemisphere, so the bias sign is
  consistent and `net_bias` pools directly. HET = real ALM silencing, WT = light artifact.
- **Bilateral ALM → reaction time (§B).** Does silencing change response latency? RT is
  zero-inflated at the acquisition floor (a lick before the reward epoch registers as 0), so it
  is split into `rt_early_frac` (early-response rate) and `rt_median` (median over rt > 0).

n is small (≤8 animals, ≤4 WT); the WT arm floors the between-genotype test, so sign /
consistency and the achievable-p floor are the evidence, not a bare p-value.

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd
from shared_setup import *

from analysis.opto import (
    compute_choice_by_stimulus, compute_side_bias, extract_opto_rt,
)
from behav_utils.analysis import paired_diff, rank_test, min_achievable_p
from plotting.opto import plot_delta_by_stimulus, plot_delta_swarm

experiment, info = load_data()
geno, groups = gather_genotypes(experiment)
het, wt = groups.get('het', []), groups.get('wt', [])
opto_ids = het + wt
if not het or not wt:
    raise RuntimeError("No HET/WT genotypes on loaded animals (gather_genotypes reads .genotype).")
print(f"loaded ({info['mode']}): het={het} wt={wt}")

## §A · Unilateral ALM → side bias

`alm_control_uni`, uniform, opto (laser at ALM) vs `opto_off`. Same hemisphere throughout, so
`net_bias` (overall ΔP(B)) is sign-consistent and pools across animals. Real ALM silencing (HET)
predicts a lateralised bias; WT (light only) should sit at ≈0. HET-Δ vs WT-Δ isolates a real
bias from the light artifact.

In [ ]:
cbs_uni = compute_choice_by_stimulus(experiment, phase='uniform',
                                     session_type='alm_control_uni', animals=opto_ids)
present = sorted(cbs_uni['binned']['animal'].unique()) if not cbs_uni['binned'].empty else []
print(f"unilateral ALM: {len(present)} animal(s) with data -> {present}")

sb_uni = compute_side_bias(cbs_uni)
display(sb_uni.round(3))

h = sb_uni[sb_uni.genotype == 'het']['net_bias'].dropna().to_numpy()
w = sb_uni[sb_uni.genotype == 'wt']['net_bias'].dropna().to_numpy()
resg = rank_test(h, w, paired=False) if len(h) and len(w) else {'statistic': np.nan, 'p': np.nan}
res_h = rank_test(h, np.zeros_like(h), paired=True) if len(h) >= 3 else {'statistic': np.nan, 'p': np.nan}
print(f"net_bias HET vs WT: U={resg['statistic']}  p={resg['p']:.3g}  "
      f"(min p={min_achievable_p('rank_sum', n1=len(h), n2=len(w)):.3g}, n_het={len(h)} n_wt={len(w)})")
print(f"net_bias HET vs 0:  p={res_h['p']:.3g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
plot_delta_by_stimulus(cbs_uni, ax=axes[0], genotype='het'); axes[0].set_title('unilateral ALM — HET')
plot_delta_by_stimulus(cbs_uni, ax=axes[1], genotype='wt');  axes[1].set_title('unilateral ALM — WT')
fig.tight_layout()

long_uni = sb_uni.melt(id_vars=['animal', 'genotype'],
                       value_vars=['net_bias', 'tail_delta', 'boundary_delta'],
                       var_name='stat', value_name='delta')
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, st in zip(axes, ['net_bias', 'tail_delta', 'boundary_delta']):
    plot_delta_swarm(long_uni, st, ax=ax, p_value=resg['p'] if st == 'net_bias' else None)
fig.tight_layout()

## §B · Bilateral ALM → reaction time

`alm_control_bi`, uniform, opto vs `opto_off`. `extract_opto_rt` pools non-abort trials per
animal×condition and returns two stats — `rt_early_frac` (fraction with `reaction_time == 0`,
the early / anticipatory licks) and `rt_median` (median over `reaction_time > 0`). The split is
forced by the data: early licks can exceed 75 % of trials, so a plain median saturates at 0.
Δ = opto − opto_off per animal, tested within genotype (vs 0) and between (HET-Δ vs WT-Δ).

In [ ]:
pts_rt = extract_opto_rt(experiment, phases='uniform', session_type='alm_control_bi',
                         trial_types=('opto', 'opto_off'), animals=opto_ids)
present = sorted(pts_rt['animal'].unique()) if not pts_rt.empty else []
print(f"bilateral ALM: {len(present)} animal(s) with RT data -> {present}")
if not pts_rt.empty:
    display(pts_rt.pivot_table(index=['animal', 'genotype'], columns=['stat', 'trial_type'],
                               values='value').round(3))

drt = (paired_diff(pts_rt, by='trial_type', a='opto', b='opto_off')  # Δ per (animal, stat)
       if not pts_rt.empty else pd.DataFrame(columns=['animal', 'genotype', 'stat', 'delta']))

In [ ]:
RT_STATS = ['rt_early_frac', 'rt_median']
rows = []
for st in RT_STATS:
    hv = drt[(drt.genotype == 'het') & (drt.stat == st)]['delta'].dropna().to_numpy()
    wv = drt[(drt.genotype == 'wt') & (drt.stat == st)]['delta'].dropna().to_numpy()
    resg = rank_test(hv, wv, paired=False) if len(hv) and len(wv) else {'statistic': np.nan, 'p': np.nan}
    res_h = rank_test(hv, np.zeros_like(hv), paired=True) if len(hv) >= 3 else {'statistic': np.nan, 'p': np.nan}
    res_w = rank_test(wv, np.zeros_like(wv), paired=True) if len(wv) >= 3 else {'statistic': np.nan, 'p': np.nan}
    rows.append(dict(stat=st, n_het=len(hv), n_wt=len(wv),
                     med_het=float(np.median(hv)) if len(hv) else np.nan,
                     med_wt=float(np.median(wv)) if len(wv) else np.nan,
                     p_het_vs_wt=resg['p'], p_het_vs_0=res_h['p'], p_wt_vs_0=res_w['p'],
                     min_p_hw=min_achievable_p('rank_sum', n1=len(hv), n2=len(wv))))
rt_tests = pd.DataFrame(rows)
display(rt_tests.round(4))

if not drt.empty:
    fig, axes = plt.subplots(1, len(RT_STATS), figsize=(4 * len(RT_STATS), 3.6), squeeze=False)
    for ax, st in zip(axes[0], RT_STATS):
        pv = rt_tests.loc[rt_tests.stat == st, 'p_het_vs_wt'].iloc[0]
        plot_delta_swarm(drt, st, ax=ax, p_value=pv)
        ax.set_ylabel(f"Δ {st} (opto − opto_off)")
    fig.tight_layout()

**Reading it.** §A: if HET `net_bias` is non-zero with a consistent sign and WT ≈ 0 (HET-Δ vs
WT-Δ separated as far as n allows), unilateral ALM silencing biases choice — a regional effect
distinct from PPC. §B: a HET-specific shift in `rt_early_frac` (more / fewer early licks) or
`rt_median` (slower / faster real responses), with WT ≈ 0, is a bilateral-ALM latency effect.
WT n floors both between-genotype tests, so weigh the achievable-p and the sign, not the bare p.
The regional-specificity claim rests on these ALM effects differing in kind from the PPC result
in `30`/`31`.